# Vinyl Label Reader + Discogs Search

Read a vinyl record label image using the backend API, which extracts album/artist metadata via vision LLM and searches **Discogs**.

**Setup:**
1. Add your keys to `.env` (`DISCOGS_TOKEN`, `OPENROUTER_API_KEY`)
2. Start the backend: `make dev`
3. Change the vision model in `backend/config.py` if needed

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

LABEL_IMAGE_PATH = Path("img/labels/columbia-black-silvewer-early-1000px.jpg")

In [ ]:
image_path = LABEL_IMAGE_PATH
mime_type = "image/jpeg" if image_path.suffix.lower() in (".jpg", ".jpeg") else "image/png"

with open(image_path, "rb") as f:
    resp = requests.post(
        f"{BACKEND_URL}/api/search",
        files={"file": (image_path.name, f, mime_type)},
        data={"media_type": "vinyl"},
    )
resp.raise_for_status()
data = resp.json()

print("Label data:")
print(json.dumps(data["label_data"], indent=2))
print(f"\nStrategy: {data['strategy']}")
print(f"Total results: {data['total']}")

if data.get("debug"):
    debug = data["debug"]
    if "strategies_tried" in debug:
        print(f"Strategies tried: {debug['strategies_tried']}")
    if "timing_ms" in debug:
        print(f"Timing: {debug['timing_ms']}")
    print(f"Cache hit: {debug.get('cache_hit', 'N/A')}")

In [ ]:
releases = data["results"]

if releases:
    top = releases[0]
    print(f"Best match: {top['title']}")
    print(f"  catno={top.get('catno')}, country={top.get('country')}, year={top.get('year')}")
    if top.get("discogs_url"):
        print(f"  {top['discogs_url']}")
else:
    print("No results found.")

In [ ]:
from IPython.display import display

if not releases:
    print("No releases found — nothing to display.")
    df = pd.DataFrame()
else:
    df = pd.DataFrame([
        {
            "title": r.get("title"),
            "year": r.get("year"),
            "country": r.get("country"),
            "format": r.get("format"),
            "label": r.get("label"),
            "catno": r.get("catno"),
            "discogs_url": r.get("discogs_url"),
            "cover_image": r.get("cover_image"),
        }
        for r in releases
    ])

    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df.sort_values("year", inplace=True, na_position="last")
    df.reset_index(drop=True, inplace=True)
    display(df)

---

# Batch Run Analysis

Query MongoDB for batch processing results and analyze success/failure rates.

In [ ]:
from pymongo import MongoClient

mongo_uri = os.getenv("MONGO_URI", "mongodb://localhost:27017")
mongo_db = os.getenv("MONGO_DB", "vinyl_recko")
client = MongoClient(mongo_uri)
db = client[mongo_db]

# Overview
batches = list(db.batches.find({}, {"_id": 0}))
items = list(db.batch_items.find({}, {"_id": 0}))

print(f"Batches: {len(batches)}")
for b in batches:
    print(f"  {b['batch_id'][:8]}... — {b['status']}, {b['processed']}/{b['total_images']} processed, {b['failed']} failed")

print(f"\nTotal items: {len(items)}")
status_counts = pd.Series([i["status"] for i in items]).value_counts()
review_counts = pd.Series([i["review_status"] for i in items]).value_counts()
print(f"\nBy processing status:\n{status_counts.to_string()}")
print(f"\nBy review status:\n{review_counts.to_string()}")

In [ ]:
# Strategy distribution
strategies = [i.get("strategy", "N/A") for i in items if i["status"] == "completed"]
strategy_series = pd.Series(strategies)

# Simplify strategy names for grouping
def simplify_strategy(s):
    if not s:
        return "none"
    if s.startswith("catno="):
        return "catno"
    if "track names" in s:
        return "track names"
    if s.startswith("q="):
        return "freeform (q)"
    if s.startswith("release_title="):
        return "release_title + artist"
    if "fuzzy" in s:
        return "artist + fuzzy title"
    if "all releases" in s:
        return "artist only (no title match)"
    return s

strategy_groups = strategy_series.map(simplify_strategy).value_counts()
print("Strategy distribution (completed items):")
print(strategy_groups.to_string())
print(f"\nTotal completed: {len(strategies)}")

In [ ]:
# Problematic items: dismissed, no results, errored
problems = []
for i in items:
    result_count = len(i.get("results") or [])
    label = i.get("label_data") or {}
    artists = ", ".join(label.get("artists") or [])
    albums = ", ".join(label.get("albums") or [])
    row = {
        "filename": i["image_filename"],
        "status": i["status"],
        "review": i["review_status"],
        "strategy": simplify_strategy(i.get("strategy")),
        "results": result_count,
        "artists": artists,
        "albums": albums,
        "error": i.get("error"),
    }
    if i["review_status"] == "skipped" or result_count == 0 or i["status"] == "error":
        problems.append(row)

problems_df = pd.DataFrame(problems)
print(f"Problematic items: {len(problems_df)} / {len(items)}")
display(problems_df[["filename", "status", "review", "strategy", "results", "artists", "albums", "error"]])

In [ ]:
# Telemetry: search_records timing
records = list(db.search_records.find({}, {"_id": 0}))
if records:
    records_df = pd.DataFrame(records)
    batch_records = records_df[records_df["batch_id"].notna()]
    single_records = records_df[records_df["batch_id"].isna()]

    print(f"Search records: {len(records_df)} total ({len(batch_records)} batch, {len(single_records)} single)")
    print(f"\nBatch timing (ms):")
    if not batch_records.empty:
        print(f"  mean: {batch_records['total_duration_ms'].mean():.0f}")
        print(f"  median: {batch_records['total_duration_ms'].median():.0f}")
        print(f"  min: {batch_records['total_duration_ms'].min():.0f}")
        print(f"  max: {batch_records['total_duration_ms'].max():.0f}")
    print(f"\nBatch status distribution:")
    print(batch_records["status"].value_counts().to_string())
else:
    print("No search records found.")